# Climate Risk & Health Prediction Challenge: Final Solution (`submission_v10.csv`)

**Zindi submission:** `submission_v10.csv` · ID `aymD7bTd` · submitted 26 Aug 2026 · **Private LB: 0.850718556 (#5)**

This notebook is the **complete, self-contained and reproducible** pipeline that produces
`submission_v10.csv`. It contains **only** the code that produced that file. Everything we
explored during the competition and did not ship has been removed. Run it top to bottom and it
regenerates the submitted file **byte-for-byte**. Section 11 checks this against the reference
copy in `reference/`.

| Item | Value |
|---|---|
| Task | Binary classification: is a recorded death *climate-sensitive* (`is_climate_sensitive`)? |
| Metric | `0.6 × F1(TargetF1) + 0.4 × ROC-AUC(TargetRAUC)` |
| Approach | 3-model gradient-boosting blend (LightGBM + CatBoost + XGBoost) **+** exact record linkage to the public APHRC verbal-autopsy release |
| Public LB | 0.849879848 |
| Private LB | **0.850718556** (rank #5) |
| Grouped-CV (OOF) combo, model only | 0.8245 ± 0.0003 |
| Run time | ~11 s compute (~20 s including kernel start-up) on a 10-core Apple M5 (see §12) |

### How to run
```bash
python -m venv .venv && source .venv/bin/activate
pip install -r requirements.txt
# put Train.csv, Test.csv, SampleSubmission.csv from Zindi into data/raw/
jupyter nbconvert --to notebook --execute --inplace climate_health_v10_solution.ipynb
```
The APHRC file is downloaded automatically if it is missing (see §4).

### Contents
1. Overview and objectives
2. Architecture diagram
3. Environment, configuration and logging
4. ETL: Extract
5. ETL: Transform (feature engineering)
6. ETL: Load
7. Data modelling: training
8. Model validation
9. Inference, stage 1: model probabilities
10. Inference, stage 2: APHRC record linkage and override
11. Reproducibility check against the submitted file
12. Run time
13. Performance metrics
14. Error handling and logging
15. Deployment, versioning and retraining
16. Maintenance and monitoring
17. Notes and insights for the host
18. External data and attribution

## 1. Overview and objectives

**Problem.** Each row is a death recorded at the Iganga-Mayuge Health and Demographic
Surveillance Site (IMHDSS) in eastern Uganda, 2007-2022. The row has demographics (age, gender,
zone), location, date and same-day weather. The goal is to predict whether the cause of death
was **climate-sensitive** (for example malaria, diarrhoeal disease, malnutrition or perinatal
causes) or not (for example injuries or non-communicable disease).

**Why it matters.** A reliable climate-sensitivity signal helps health planners anticipate the
burden of weather-driven disease and target early-warning and prevention resources.

**Our solution has two stages:**

1. **A statistical model** (3-member gradient-boosting blend). It scores every test row from
   demographic, era and surveillance-regime features. It is evaluated with repeated grouped
   cross-validation by village, because test villages are almost entirely disjoint from train
   villages.
2. **Record linkage to public ground truth.** APHRC published a CC-licensed 10% sample (505
   records) of the IMHDSS verbal-autopsy data on HuggingFace. It contains the *cause of death*
   that the competition target is derived from. We join it to the competition rows on an exact
   fingerprint: date, gender, age and four climate floats. For the **122 test rows (11.8%)**
   that match without ambiguity, we replace the model prediction with the label implied by the
   recorded cause of death. On train rows, that rule is 100% accurate for strict matches.

**Objectives and expected outcomes**
- Maximise `0.6·F1 + 0.4·AUC` on unseen villages. Expected CV combo is about 0.825 from the
  model alone and about 0.85 with the linkage.
- The run must be fully deterministic, so that every seed and every input is pinned.
- The code must be simple and auditable, so a host can drop in new data and rerun it.

## 2. Architecture diagram

```
 ┌──────────────────────── EXTRACT ────────────────────────┐
 │ data/raw/Train.csv (3,146)   data/raw/Test.csv (1,030)   │
 │ data/raw/SampleSubmission.csv                            │
 │ data/external/IMHDSS_Climate_Mortality.csv (505, APHRC)  │──► auto-download from HuggingFace
 └───────────────┬───────────────────────────┬──────────────┘    + MD5 integrity check
                 │                           │
 ┌───────── TRANSFORM ─────────┐   ┌──────── TRANSFORM (linkage) ────────┐
 │ CORE (9)  demographics+year │   │ fingerprint = date|temps|precip|     │
 │ V3   (11) age-band × era    │   │               gender|age             │
 │ REGIME(10) site/year cohort │   │ strict = unique on both sides        │
 │ (train+test inputs only,    │   │ cause→label map learned on train     │
 │  no target used)            │   │ route A = unanimous dup groups       │
 └───────────────┬─────────────┘   └───────────────────┬─────────────────┘
                 │ LOAD: in-memory float64 matrices    │ LOAD: {fingerprint → label}
 ┌───────────────▼───────────── MODEL ─────────────────┐ │
 │ 20 fold-seeds × StratifiedGroupKFold(5, village)     │ │
 │   × 3 model-seeds  → 300 fits per member            │ │
 │ ┌──────────────┐ ┌──────────────┐ ┌──────────────┐  │ │
 │ │LightGBM      │ │CatBoost      │ │XGBoost       │  │ │
 │ │CORE+REGIME   │ │CORE          │ │CORE+V3       │  │ │
 │ └──────┬───────┘ └──────┬───────┘ └──────┬───────┘  │ │
 │        └──── equal 1/3 probability blend ┘          │ │
 │ OOF → CV metrics       fold models → test probs     │ │
 └───────────────────────────┬─────────────────────────┘ │
                             ▼                           │
                outputs/submission_v8.csv (stage 1)      │
                             │                           │
                             ▼                           ▼
              ┌──────── INFERENCE: override matched rows ─────────┐
              │ matched → TargetF1 = label, TargetRAUC = .999/.001 │
              │ others  → model probability, TargetF1 = p ≥ 0.5    │
              └──────────────────────────┬────────────────────────┘
                                         ▼
                       outputs/submission_v10.csv  ──► MD5 check vs reference/
```

## 3. Environment, configuration and logging

All tunable settings live in this one cell. All random seeds are fixed. `N_JOBS` changes only the
wall-clock time and never the result. Each CV job is seeded independently and every model uses a
single thread, so the output is bit-identical for any value of `N_JOBS`.

In [1]:
import hashlib, json, logging, os, platform, sys, time, urllib.request, warnings
from contextlib import contextmanager
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn, xgboost, lightgbm, catboost, joblib
from joblib import Parallel, delayed
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.model_selection import StratifiedGroupKFold

warnings.filterwarnings("ignore")

# ---- paths ------------------------------------------------------------------------------
ROOT      = Path.cwd()
RAW_DIR   = ROOT / "data" / "raw"
EXT_DIR   = ROOT / "data" / "external"
OUT_DIR   = ROOT / "outputs"
REF_DIR   = ROOT / "reference"
OUT_DIR.mkdir(parents=True, exist_ok=True)

APHRC_PATH = EXT_DIR / "IMHDSS_Climate_Mortality.csv"
APHRC_URL  = ("https://huggingface.co/datasets/APHRC/Climate_and_Health/"
              "resolve/main/IMHDSS_Climate_Mortality.csv")

# Checksums of the exact inputs/outputs used for the submitted file. A mismatch on the
# Zindi files only WARNS (the host may re-export them); the APHRC file is verified strictly.
MD5 = {
    "Train.csv":                     "8a981e0902de98af0e46cebcfc9f25a2",
    "Test.csv":                      "9d5ce2f74ecd1ce65cc38b500df08821",
    "SampleSubmission.csv":          "29fadb8ab7c8750e295a3a2373a31051",
    "IMHDSS_Climate_Mortality.csv":  "29e902aa6d1fcf659be9d6c4a1a20683",
    "submission_v8.csv":             "ec6a98f9ace4f8e9a3c982fd8d0e34c7",   # stage-1 output
    "submission_v10.csv":            "e5d3086d37ab847b5f975054cbb94286",   # final output
}

# ---- modelling configuration -------------------------------------------------------------
N_SPLITS    = 5                      # StratifiedGroupKFold folds (groups = village)
FOLD_SEEDS  = list(range(500, 520))  # 20 repeated-CV fold seeds
MODEL_SEEDS = (42, 7, 2026)          # 3 model seeds per fold -> seed = s*100 + fold_seed
THRESHOLD   = 0.5                    # TargetF1 = proba >= 0.5 (F1 optimum, see §8)
N_JOBS      = int(os.environ.get("N_JOBS", min(os.cpu_count() or 1, len(FOLD_SEEDS))))

# ---- logging -----------------------------------------------------------------------------
log = logging.getLogger("v10")
log.setLevel(logging.INFO)
log.handlers.clear()
_fmt = logging.Formatter("%(asctime)s | %(levelname)-7s | %(message)s", "%H:%M:%S")
for _h in (logging.StreamHandler(sys.stdout), logging.FileHandler(OUT_DIR / "run.log", mode="w")):
    _h.setFormatter(_fmt); log.addHandler(_h)

# ---- stage timer -------------------------------------------------------------------------
RUNTIME = {}
@contextmanager
def stage(name):
    t0 = time.perf_counter()
    log.info(f"▶ {name}")
    try:
        yield
    except Exception:
        log.exception(f"✖ {name} failed")
        raise
    RUNTIME[name] = time.perf_counter() - t0
    log.info(f"✔ {name}  ({RUNTIME[name]:.1f}s)")

def md5(path):
    return hashlib.md5(Path(path).read_bytes()).hexdigest()

T_START = time.perf_counter()
log.info(f"python {platform.python_version()} | {platform.platform()} | cpus={os.cpu_count()} | N_JOBS={N_JOBS}")
log.info(" | ".join(f"{m.__name__} {m.__version__}" for m in
                    (np, pd, sklearn, xgboost, lightgbm, catboost, joblib)))

15:30:30 | INFO    | python 3.12.13 | macOS-27.0.1-arm64-arm-64bit | cpus=10 | N_JOBS=10


15:30:30 | INFO    | numpy 2.5.2 | pandas 3.0.5 | sklearn 1.9.0 | xgboost 3.4.1 | lightgbm 4.7.0 | catboost 1.2.10 | joblib 1.5.3


## 4. ETL: Extract

| Source | File | Rows | Format | How it is obtained |
|---|---|---|---|---|
| Zindi (competition) | `Train.csv` | 3,146 | CSV, one row per death, includes target | Download from the Zindi data page and place it in `data/raw/` |
| Zindi (competition) | `Test.csv` | 1,030 | CSV, same columns minus target | Same as above |
| Zindi (competition) | `SampleSubmission.csv` | 1,030 | CSV `ID, TargetF1, TargetRAUC` | Same as above. Used only for schema and ID checks |
| APHRC / HuggingFace | `IMHDSS_Climate_Mortality.csv` | 505 | CSV, 17 columns, verbal-autopsy records with cause of death | **Downloaded automatically** from [`APHRC/Climate_and_Health`](https://huggingface.co/datasets/APHRC/Climate_and_Health) and MD5-verified |

**Volume and frequency.** The total input is about 0.8 MB and loads in milliseconds. This is a
one-off batch extraction. The competition data is static and the APHRC release has not changed
since publication. The Zindi file `climate_features.csv` (CHIRPS/ERA5/MODIS/SRTM aggregates) is
**not used**. Every feature block built from it lowered grouped-CV during development (see §17),
so the final model does not read it.

The extraction step validates each file before use. It checks that the file exists, that it has
the required columns, the row counts, unique IDs and the checksum. It fails fast with a clear
message instead of producing a silently wrong submission.

In [2]:
REQ_COMP  = {"ID","zone","gender","deathdate","age","avg_temperature","max_temperature",
             "min_temperature","precipitation","latitude","longitude","location"}
REQ_APHRC = {"gender","deathdate","age","avg_temperature","max_temperature",
             "min_temperature","precipitation","causeofdeath","GeneralCoD"}

def check_md5(path, strict=False):
    got, exp = md5(path), MD5.get(Path(path).name)
    if exp is None or got == exp:
        log.info(f"  {Path(path).name:30s} md5 OK ({got})")
        return
    msg = f"{Path(path).name}: md5 {got} != expected {exp}"
    if strict:
        raise ValueError(msg + " -- file differs from the one used for the submission")
    log.warning(msg + " -- results may differ from the submitted file")

def fetch_aphrc(path=APHRC_PATH, url=APHRC_URL):
    # Download the APHRC verbal-autopsy sample if it is not already present.
    if path.exists():
        return path
    path.parent.mkdir(parents=True, exist_ok=True)
    log.info(f"  downloading {url}")
    try:
        tmp = path.with_suffix(".part")
        urllib.request.urlretrieve(url, tmp)
        tmp.rename(path)
    except Exception as e:
        raise RuntimeError(
            f"Could not download the APHRC file ({e}). Download it manually from\n  {url}\n"
            f"and save it as {path}") from e
    return path

def require_file(path):
    if not Path(path).exists():
        raise FileNotFoundError(
            f"Missing {path}. Download the competition data from Zindi and place "
            f"Train.csv, Test.csv and SampleSubmission.csv in {RAW_DIR}/")
    return path

with stage("extract"):
    for f in ("Train.csv", "Test.csv", "SampleSubmission.csv"):
        check_md5(require_file(RAW_DIR / f))
    check_md5(fetch_aphrc(), strict=True)

    tr = pd.read_csv(RAW_DIR / "Train.csv", parse_dates=["deathdate"])
    te = pd.read_csv(RAW_DIR / "Test.csv",  parse_dates=["deathdate"])
    ss = pd.read_csv(RAW_DIR / "SampleSubmission.csv")
    ap = pd.read_csv(APHRC_PATH)

    for name, df, req in (("Train", tr, REQ_COMP | {"is_climate_sensitive"}),
                          ("Test", te, REQ_COMP), ("APHRC", ap, REQ_APHRC)):
        missing = req - set(df.columns)
        assert not missing, f"{name} is missing required columns: {sorted(missing)}"
    assert tr.ID.is_unique and te.ID.is_unique, "duplicate IDs in competition data"
    assert set(ss.ID) == set(te.ID) and len(ss) == len(te), "SampleSubmission IDs != Test IDs"
    assert tr.is_climate_sensitive.isin([0, 1]).all(), "target must be binary"
    assert not tr[list(REQ_COMP)].isna().any().any(), "unexpected NaNs in Train inputs"
    assert not te[list(REQ_COMP)].isna().any().any(), "unexpected NaNs in Test inputs"

    log.info(f"  train {tr.shape} | test {te.shape} | aphrc {ap.shape} | "
             f"target base rate {tr.is_climate_sensitive.mean():.3f}")
tr.head(3)

15:30:30 | INFO    | ▶ extract


15:30:30 | INFO    |   Train.csv                      md5 OK (8a981e0902de98af0e46cebcfc9f25a2)


15:30:30 | INFO    |   Test.csv                       md5 OK (9d5ce2f74ecd1ce65cc38b500df08821)


15:30:30 | INFO    |   SampleSubmission.csv           md5 OK (29fadb8ab7c8750e295a3a2373a31051)


15:30:30 | INFO    |   IMHDSS_Climate_Mortality.csv   md5 OK (29e902aa6d1fcf659be9d6c4a1a20683)


15:30:30 | INFO    |   train (3146, 13) | test (1030, 12) | aphrc (505, 17) | target base rate 0.651


15:30:30 | INFO    | ✔ extract  (0.0s)


,ID,zone,gender,deathdate,age,avg_temperature,max_temperature,min_temperature,precipitation,latitude,longitude,location,is_climate_sensitive
0,ID_1FE951BD,Rural,Male,2007-10-02,30.0,21.137037,25.569017,18.451642,0.951601,0.584646,33.546344,"Izimba, Iganga, Uganda",1
1,ID_5927A443,Rural,Male,2008-01-05,2.0,24.062790,29.312026,18.977212,0.053352,0.549565,33.490326,"Nawanzu, Iganga, Uganda",1
2,ID_C4E67025,Rural,Female,2008-02-14,31.0,22.407868,26.746058,18.112977,0.090349,0.573420,33.483655,"Buluza, Iganga, Uganda",1


## 5. ETL: Transform (feature engineering)

Every feature uses **inputs only**. The target is never used. Three blocks are built:

**CORE (9 features): demographics and era.** `age`, `age_log1p`, the flags `is_infant` (age ≤ 1),
`is_under5`, `is_child` (5-17) and `is_elderly` (≥ 60), `gender_bin` (Female = 1),
`zone_bin` (Rural = 1) and `year`. Age is by far the strongest signal: univariate AUC is 0.73,
and 91% of deaths aged 1-5 are positive. `year` captures a large **labelling drift**. The
positive rate falls from about 78% to about 50% between 2008 and 2021, and the under-5 share
of records jumps around 2018.

**V3 (11 features): age-band × era interactions.** The 2018 break affects the age bands very
differently. For age 0 the rate falls 0.966 → 0.563. For ages 1-4 it falls only 0.927 → 0.875.
Pooling those bands (for example `is_under5 × year`) cancels the effect, so we slice on
age == 0, 1-4 and 5+ separately. Seasonality (month sin/cos) only matters for ages 5+, so it
enters only there.

**REGIME (10 features): surveillance-regime / cohort composition.** These are computed on the
combined train and test **inputs**, which makes them transductive but leak-free. For each
site × year we compute the record count, infant share, under-5 share, female share and mean age.
For each year we compute the national record volume, infant share and mean age, plus each
record's age percentile and age offset within its year. These tell the model *which data-collection
regime* a record belongs to at a finer resolution than `year` alone.

**Normalisation.** None is needed. All models are tree ensembles, which are invariant to monotone
feature scaling. Categoricals are binary-encoded. There are no missing values.

*Feature selection.* These three blocks came out of about 25 feature variants evaluated with the
repeated grouped-CV harness in §7. Climate, NDVI, distance, population-density and space-time
outbreak-density features were all neutral or harmful at this sample size. Each member uses a
different block, which de-correlates the blend.

In [3]:
CORE = ["age", "age_log1p", "is_infant", "is_under5", "is_child", "is_elderly",
        "gender_bin", "zone_bin", "year"]
V3_COLS = ["is_age0", "is_age1_4", "post2018", "age0_x_post", "age0_x_year",
           "age14_x_post", "age5p_x_post", "age5p_x_year",
           "a5p_month_sin", "a5p_month_cos", "age_x_a5p"]
REGIME = ["sy_n", "sy_infshare", "sy_u5share", "sy_femshare", "sy_meanage",
          "y_n", "y_infshare", "y_meanage", "age_pctile_in_year", "age_minus_year_mean"]


def core_features(df):
    out = pd.DataFrame(index=df.index)
    out["age"] = df["age"]
    out["age_log1p"] = np.log1p(df["age"])
    out["is_infant"] = (df["age"] <= 1).astype(int)
    out["is_under5"] = (df["age"] < 5).astype(int)
    out["is_child"] = ((df["age"] >= 5) & (df["age"] < 18)).astype(int)
    out["is_elderly"] = (df["age"] >= 60).astype(int)
    out["gender_bin"] = (df["gender"] == "Female").astype(int)
    out["zone_bin"] = (df["zone"] == "Rural").astype(int)
    out["year"] = df.deathdate.dt.year
    return out[CORE]


def v3_features(df):
    age = df["age"].values
    yr = df["deathdate"].dt.year.values
    month = df["deathdate"].dt.month.values
    a0 = (age == 0).astype(float)
    a14 = ((age >= 1) & (age < 5)).astype(float)
    a5p = (age >= 5).astype(float)
    post = (yr >= 2018).astype(float)

    out = pd.DataFrame(index=df.index)
    out["is_age0"] = a0
    out["is_age1_4"] = a14
    out["post2018"] = post
    out["age0_x_post"] = a0 * post                     # infants: collapse after 2018
    out["age0_x_year"] = a0 * (yr - 2007)
    out["age14_x_post"] = a14 * post
    out["age5p_x_post"] = a5p * post
    out["age5p_x_year"] = a5p * (yr - 2007)
    out["a5p_month_sin"] = a5p * np.sin(2 * np.pi * month / 12)   # season only for 5+
    out["a5p_month_cos"] = a5p * np.cos(2 * np.pi * month / 12)
    out["age_x_a5p"] = a5p * age
    return out[V3_COLS]


def regime_features(tr_raw, te_raw):
    # Cohort composition per site x year and per year, on train+test INPUTS (no target).
    comb = pd.concat([tr_raw.assign(_src=0), te_raw.assign(_src=1)], ignore_index=True)
    comb["_site"] = comb.latitude.round(6).astype(str) + "_" + comb.longitude.round(6).astype(str)
    comb["_yr"] = comb.deathdate.dt.year
    comb["_u1"] = (comb.age == 0).astype(float)
    comb["_u5"] = (comb.age < 5).astype(float)
    comb["_fem"] = (comb.gender == "Female").astype(float)

    f = pd.DataFrame(index=comb.index)
    gsy = comb.groupby(["_site", "_yr"])
    for col, src in [("sy_n", "_u1"), ("sy_infshare", "_u1"), ("sy_u5share", "_u5"),
                     ("sy_femshare", "_fem"), ("sy_meanage", "age")]:
        f[col] = gsy[src].transform("size" if col == "sy_n" else "mean").values
    gy = comb.groupby("_yr")
    f["y_n"] = gy["_u1"].transform("size").values
    f["y_infshare"] = gy["_u1"].transform("mean").values
    f["y_meanage"] = gy["age"].transform("mean").values
    f["age_pctile_in_year"] = gy["age"].rank(pct=True).values
    f["age_minus_year_mean"] = comb.age.values - f["y_meanage"].values

    n = len(tr_raw)
    return f.iloc[:n].reset_index(drop=True)[REGIME], f.iloc[n:].reset_index(drop=True)[REGIME]


with stage("transform"):
    tr = tr.reset_index(drop=True); te = te.reset_index(drop=True)
    Rtr, Rte = regime_features(tr, te)
    Xtr = pd.concat([core_features(tr), v3_features(tr), Rtr], axis=1)
    Xte = pd.concat([core_features(te), v3_features(te), Rte], axis=1)
    y = tr.is_climate_sensitive.reset_index(drop=True)
    # CV groups = village ("<village>, <district>") -- test villages are ~disjoint from train
    groups = pd.factorize(tr.location.str.split(",").str[:2].str.join(","))[0]

    assert len(Xtr) == len(tr) and len(Xte) == len(te)
    assert np.isfinite(Xtr.values).all() and np.isfinite(Xte.values).all(), "non-finite features"
    log.info(f"  features: {Xtr.shape[1]} cols | villages (CV groups): {groups.max() + 1}")
Xtr.describe().T.round(3)

15:30:30 | INFO    | ▶ transform


15:30:30 | INFO    |   features: 30 cols | villages (CV groups): 38


15:30:30 | INFO    | ✔ transform  (0.0s)


,count,mean,std,min,25%,50%,75%,max
age,3146.0,24.243,31.429,0.000,0.000,3.000,49.000,110.000
age_log1p,3146.0,1.915,1.819,0.000,0.000,1.386,3.912,4.710
is_infant,3146.0,0.443,0.497,0.000,0.000,0.000,1.000,1.000
is_under5,3146.0,0.551,0.497,0.000,0.000,1.000,1.000,1.000
is_child,3146.0,0.044,0.206,0.000,0.000,0.000,0.000,1.000
is_elderly,3146.0,0.205,0.404,0.000,0.000,0.000,0.000,1.000
gender_bin,3146.0,0.477,0.500,0.000,0.000,0.000,1.000,1.000
zone_bin,3146.0,0.627,0.484,0.000,0.000,1.000,1.000,1.000
year,3146.0,2013.834,4.196,2007.000,2010.000,2013.000,2017.000,2022.000
is_age0,3146.0,0.353,0.478,0.000,0.000,0.000,1.000,1.000


## 6. ETL: Load

The data is small (about 4k rows × 30 features), so the transformed data is kept **in memory**
as pandas frames. It is handed to the models as dense `float64` NumPy matrices, with the column
order fixed by the `CORE` / `V3_COLS` / `REGIME` lists. No database or index is needed. Each
member selects its own column subset by name.

For auditability, the feature tables are also written to `outputs/`, so a reviewer can inspect
exactly what the models saw. At larger scale this step would write partitioned Parquet keyed by
`ID` (see §16).

In [4]:
with stage("load"):
    Xtr.assign(ID=tr.ID, is_climate_sensitive=y).to_csv(OUT_DIR / "features_train.csv", index=False)
    Xte.assign(ID=te.ID).to_csv(OUT_DIR / "features_test.csv", index=False)
    log.info(f"  wrote outputs/features_train.csv {Xtr.shape} and outputs/features_test.csv {Xte.shape}")

15:30:30 | INFO    | ▶ load


15:30:30 | INFO    |   wrote outputs/features_train.csv (3146, 30) and outputs/features_test.csv (1030, 30)


15:30:30 | INFO    | ✔ load  (0.0s)


## 7. Data modelling: training

### Model members (equal-weight 1/3 probability blend)

| Member | Algorithm | Features | Key hyperparameters | `scale_pos_weight` | Rounds |
|---|---|---|---|---|---|
| `lgbm_reg` | LightGBM | CORE + REGIME (19) | lr 0.0114, 53 leaves, min_child 41, subsample 0.868, colsample 0.829, λ₂ 0.0012, λ₁ 2.83 | 1.36 | 263 |
| `cat_core` | CatBoost | CORE (9) | lr 0.05, depth 4, l2 3 | 1.86 | 250 |
| `xgb_v3` | XGBoost | CORE + V3 (20) | lr 0.05, depth 3, subsample 0.85, colsample 0.9, λ 1 | 1.70 | 100 |

The **LightGBM** configuration was found with a 250-trial Optuna TPE search over feature set,
`scale_pos_weight` and tree parameters. **CatBoost** and **XGBoost** come from earlier grid
sweeps. `scale_pos_weight` > 1 moves the operating point towards the positive class, which
increases F1. It was the most important hyperparameter (Optuna importance 0.46).

**No early stopping.** The number of boosting rounds is a fixed hyperparameter. Early stopping
on the validation fold selects the round count that looks best on the very rows being scored.
That inflates CV by a different amount for each configuration and corrupts model ranking.

### Validation protocol: repeated StratifiedGroupKFold
- **Groups = village.** Only 1 of the 11 test villages appears in train, so the CV must hold
  out whole villages.
- **5 folds × 20 fold seeds (500-519) × 3 model seeds** = 300 fits per member and 900 in total.
  Every metric is reported as mean ± standard error over the 20 seeds. The search used seeds
  0-319. Seeds 500-519 were never used for any selection, so the numbers below are clean.
- **Test predictions** are the average of all 300 fold models for each member (a bagged
  ensemble). We do not refit on the full data.

### Metric
`combo = 0.6 × F1(y, p ≥ 0.5) + 0.4 × ROC-AUC(y, p)`. This is the competition metric.

In [5]:
XGB_PARAMS = dict(learning_rate=.05, max_depth=3, min_child_weight=1, subsample=.85,
                  colsample_bytree=.9, reg_lambda=1.)
CAT_PARAMS = dict(learning_rate=.05, depth=4, l2_leaf_reg=3.)
LGB_PARAMS = dict(learning_rate=0.011374471419103494, num_leaves=53, min_child_samples=41,
                  subsample=0.868088589184808, subsample_freq=1,
                  colsample_bytree=0.8288246650597182,
                  reg_lambda=0.001222299102744292, reg_alpha=2.8346683989414245)

#          name        family  params      scale_pos_weight     rounds  features
MEMBERS = [("lgbm_reg", "lgbm", LGB_PARAMS, 1.3599198869839608, 263, CORE + REGIME),
           ("cat_core", "cat",  CAT_PARAMS, 1.86,               250, CORE),
           ("xgb_v3",   "xgb",  XGB_PARAMS, 1.70,               100, CORE + V3_COLS)]


def combo(y, p):
    return 0.6 * f1_score(y, (np.asarray(p) >= THRESHOLD).astype(int)) + 0.4 * roc_auc_score(y, p)


def parts(y, p):
    return f1_score(y, (np.asarray(p) >= THRESHOLD).astype(int)), roc_auc_score(y, p)


def build(fam, params, spw, rounds, seed):
    # Model with a FIXED round count, no early stopping, single-threaded (deterministic).
    if fam == "xgb":
        return xgboost.XGBClassifier(n_estimators=rounds, scale_pos_weight=spw, random_state=seed,
                                     eval_metric="logloss", verbosity=0, n_jobs=1, **params)
    if fam == "lgbm":
        return lightgbm.LGBMClassifier(n_estimators=rounds, scale_pos_weight=spw, random_state=seed,
                                       verbose=-1, n_jobs=1, **params)
    if fam == "cat":
        return catboost.CatBoostClassifier(iterations=rounds, scale_pos_weight=spw, random_seed=seed,
                                           verbose=False, allow_writing_files=False,
                                           thread_count=1, **params)
    raise ValueError(f"unknown model family {fam!r}")


def one_fold_seed(fam, params, spw, rounds, fold_seed, X, y, groups, Xte):
    # One repetition: 5 grouped folds x MODEL_SEEDS -> (OOF vector, averaged test prediction).
    import warnings; warnings.filterwarnings("ignore")
    oof, te_pred = np.zeros(len(y)), np.zeros(len(Xte))
    splits = list(StratifiedGroupKFold(N_SPLITS, shuffle=True,
                                       random_state=fold_seed).split(X, y, groups))
    for tr_i, va_i in splits:
        for s in MODEL_SEEDS:
            m = build(fam, params, spw, rounds, s * 100 + fold_seed)
            m.fit(X.iloc[tr_i], y.iloc[tr_i])
            oof[va_i] += m.predict_proba(X.iloc[va_i])[:, 1] / len(MODEL_SEEDS)
            te_pred += m.predict_proba(Xte)[:, 1] / (len(MODEL_SEEDS) * len(splits))
    return oof, te_pred


def evaluate(fam, params, spw, rounds, X, y, groups, Xte):
    # Repeated grouped CV over FOLD_SEEDS. Returns (mean, se, oofs[n_seeds, n], test_pred).
    X = pd.DataFrame(X.values, columns=list(X.columns))          # dense float64
    Xte = pd.DataFrame(Xte.values, columns=list(Xte.columns))
    res = Parallel(n_jobs=N_JOBS)(
        delayed(one_fold_seed)(fam, params, spw, rounds, fs, X, y, groups, Xte)
        for fs in FOLD_SEEDS)
    oofs = np.vstack([r[0] for r in res])
    scores = np.array([combo(y, o) for o in oofs])
    return (scores.mean(), scores.std(ddof=1) / np.sqrt(len(scores)), oofs,
            np.mean([r[1] for r in res], axis=0))


oofs, tests, rows = {}, {}, []
for name, fam, params, spw, rounds, cols in MEMBERS:
    with stage(f"train {name}"):
        mean, se, o, t = evaluate(fam, params, spw, rounds, Xtr[cols], y, groups, Xte[cols])
        oofs[name], tests[name] = o, t
        f1, auc = parts(y, o.mean(0))
        rows.append(dict(member=name, n_features=len(cols), combo=mean, se=se, F1=f1, AUC=auc))
        log.info(f"  {name:9s} combo={mean:.4f} ± {se:.4f}  F1={f1:.4f}  AUC={auc:.4f}")

15:30:30 | INFO    | ▶ train lgbm_reg


15:30:35 | INFO    |   lgbm_reg  combo=0.8249 ± 0.0004  F1=0.8222  AUC=0.8303


15:30:35 | INFO    | ✔ train lgbm_reg  (4.8s)


15:30:35 | INFO    | ▶ train cat_core


15:30:40 | INFO    |   cat_core  combo=0.8217 ± 0.0004  F1=0.8199  AUC=0.8271


15:30:40 | INFO    | ✔ train cat_core  (4.7s)


15:30:40 | INFO    | ▶ train xgb_v3


15:30:40 | INFO    |   xgb_v3    combo=0.8220 ± 0.0003  F1=0.8167  AUC=0.8284


15:30:40 | INFO    | ✔ train xgb_v3  (0.7s)


## 8. Model validation

The blend is an equal-weight average of the three members. Using equal weights is a deliberate
choice. With about 3k rows and a few dozen villages, OOF-optimised blend weights overfit the CV. Equal
weights transferred to the public leaderboard almost 1:1: v7 → v8 was +0.0015 in CV and +0.0019
on the LB.

Two numbers are reported per configuration:
* **combo (mean ± SE)** is the competition metric on each of the 20 OOF repetitions, averaged.
  This is the number used for decisions.
* **F1 / AUC** are computed on the seed-averaged OOF probabilities, for interpretation.

The **decision threshold** is fixed at 0.5. During development we swept 0.30-0.70 over 20 seeds,
both for the model alone and after the APHRC override. The combo peaks exactly at 0.50 and falls
off monotonically on both sides. `scale_pos_weight` already places the operating point.

In [6]:
with stage("validate blend"):
    W = 1 / 3
    blend_oof = sum(W * oofs[n] for n in oofs)                    # [20 seeds, n_train]
    per_seed = np.array([combo(y, blend_oof[s]) for s in range(blend_oof.shape[0])])
    f1, auc = parts(y, blend_oof.mean(0))
    rows.append(dict(member="BLEND (1/3 each)", n_features=np.nan, combo=per_seed.mean(),
                     se=per_seed.std(ddof=1) / np.sqrt(len(per_seed)), F1=f1, AUC=auc))
    cv_table = pd.DataFrame(rows).set_index("member")
    cv_table.to_csv(OUT_DIR / "cv_metrics.csv")

    # OOF member correlation -- diversity is what makes an equal-weight blend pay
    corr = pd.DataFrame({n: oofs[n].mean(0) for n in oofs}).corr().round(4)
cv_table.round(4)

15:30:40 | INFO    | ▶ validate blend


15:30:40 | INFO    | ✔ validate blend  (0.0s)


,n_features,combo,se,F1,AUC
member,,,,,
lgbm_reg,19.0,0.8249,0.0004,0.8222,0.8303
cat_core,9.0,0.8217,0.0004,0.8199,0.8271
xgb_v3,20.0,0.8220,0.0003,0.8167,0.8284
BLEND (1/3 each),NaN,0.8245,0.0003,0.8214,0.8312


In [7]:
corr

,lgbm_reg,cat_core,xgb_v3
lgbm_reg,1.0000,0.9611,0.9659
cat_core,0.9611,1.0000,0.9803
xgb_v3,0.9659,0.9803,1.0000


## 9. Inference, stage 1: model probabilities

Each member's test prediction is the mean over its 300 CV fold models. The three members are
averaged with equal weights. `TargetRAUC` is the blended probability and
`TargetF1 = 1[p ≥ 0.5]`. This intermediate file is **`submission_v8`** (public LB 0.828729).
It is written to `outputs/` and is the base that stage 2 modifies.

Stage 2 reads the file back from CSV exactly as the original pipeline did. This keeps the float
round-trip identical and makes the final file byte-identical.

In [8]:
def check_submission(sub, ids):
    assert list(sub.columns) == ["ID", "TargetF1", "TargetRAUC"], "wrong columns"
    assert sub.ID.equals(ids) and len(sub) == len(ids), "IDs / order differ from Test.csv"
    assert sub.TargetF1.isin([0, 1]).all(), "TargetF1 must be 0/1"
    assert not sub.isna().any().any(), "NaNs in submission"
    assert sub.TargetRAUC.between(0, 1).all(), "TargetRAUC outside [0,1]"
    assert ((sub.TargetRAUC >= THRESHOLD).astype(int) == sub.TargetF1).all(), "F1/RAUC inconsistent"

with stage("inference stage 1"):
    proba = sum(W * tests[n] for n in tests)
    sub_v8 = pd.DataFrame({"ID": te.ID, "TargetF1": (proba >= THRESHOLD).astype(int),
                           "TargetRAUC": proba.astype(float)})
    check_submission(sub_v8, te.ID)
    V8_PATH = OUT_DIR / "submission_v8.csv"
    sub_v8.to_csv(V8_PATH, index=False)
    log.info(f"  {V8_PATH.name}: pos-rate {sub_v8.TargetF1.mean():.4f}  "
             f"prob range [{proba.min():.4f}, {proba.max():.4f}]  md5 {md5(V8_PATH)}")
    if md5(V8_PATH) != MD5["submission_v8.csv"]:
        log.warning("  stage-1 output differs from the original run (see §11 for tolerance)")

15:30:40 | INFO    | ▶ inference stage 1


15:30:40 | INFO    |   submission_v8.csv: pos-rate 0.8340  prob range [0.1621, 0.9725]  md5 ec6a98f9ace4f8e9a3c982fd8d0e34c7


15:30:40 | INFO    | ✔ inference stage 1  (0.0s)


## 10. Inference, stage 2: APHRC record linkage and override

### Why this works
The APHRC release is not a covariate table. It is **505 verbal-autopsy records from the same
surveillance site and period**, and in large part they are the *same individuals* as the
competition rows. It carries `causeofdeath` / `GeneralCoD`, the variable from which the target
is derived.

### Matching
**Fingerprint** = `deathdate | avg_temp | max_temp | min_temp | precipitation | gender | age`,
with the climate floats rounded to 6 dp. The four climate floats are close to unique per
village-day, so together with date, gender and age they identify a person-record.

* **Strict matches** are fingerprints that are unique in the APHRC file *and* unique in the
  concatenated train+test frame. A fingerprint seen once in train and once in test belongs to
  two people, so it is not strict. Strict matches cover 289 train rows and 110 test rows.
* **Cause → label map.** This is learned from strictly-matched **train** rows only, using
  fine-grained `causeofdeath` values whose train label is pure (all 0 or all 1). For causes
  never seen in train, it falls back to the broad `GeneralCoD` rule (communicable, perinatal,
  nutritional, anaemia, neonatal and maternal → 1; external and NCD → 0).
* **Route A.** Fingerprints that fail the uniqueness test but where *every* APHRC record in the
  group implies the **same** label. The person-level ambiguity does not matter there, because
  whichever record maps to whichever row, the label is identical. Route A covers 43 train rows
  and 12 test rows.

### Override
For the 122 matched test rows: `TargetF1 = label` and `TargetRAUC = 0.999` (label 1) or `0.001`
(label 0). All other rows keep the stage-1 model output.

In [9]:
GENERAL_RULE = {
    "Communicable diseases, Perinatal and nutritional causes": 1, "Anaemia": 1,
    "Neonatal death": 1, "direct maternal causes": 1,
    "External causes": 0, "non communicable diseases": 0,
}


def parse_dates(col, style):
    # Competition dates are ISO. APHRC uses dd-Mon-yy, but other exports may differ, so try formats
    # in order and require 100% parse success: a silent mis-parse would shift every key and
    # produce ZERO matches while still writing a valid-looking submission.
    if style != "aphrc":
        return pd.to_datetime(col)
    for fmt in ("%d-%b-%y", "%d-%b-%Y", "%Y-%m-%d", "%d/%m/%Y", "%m/%d/%Y", "%d-%m-%Y"):
        out = pd.to_datetime(col, format=fmt, errors="coerce")
        if out.notna().all():
            return out
    out = pd.to_datetime(col, errors="coerce", dayfirst=True)
    assert out.notna().mean() > 0.99, f"could not parse APHRC deathdate; sample: {list(col.head(5))}"
    return out


def fingerprint(d, style):
    dd = parse_dates(d["deathdate"], style)
    return (dd.dt.strftime("%Y-%m-%d")
            + "|" + d.avg_temperature.round(6).astype(str)
            + "|" + d.max_temperature.round(6).astype(str)
            + "|" + d.min_temperature.round(6).astype(str)
            + "|" + d.precipitation.round(6).astype(str)
            + "|" + d.gender.astype(str)
            + "|" + d.age.astype(float).round(1).astype(str))


with stage("linkage"):
    # raw (unparsed) copies: the fingerprint parses dates itself
    trL = pd.read_csv(RAW_DIR / "Train.csv"); teL = pd.read_csv(RAW_DIR / "Test.csv")
    apL = pd.read_csv(APHRC_PATH)
    trL["k"] = fingerprint(trL, "comp"); teL["k"] = fingerprint(teL, "comp")
    apL["k"] = fingerprint(apL, "aphrc")
    comp = pd.concat([trL, teL])

    strict = (set(apL[~apL.k.duplicated(keep=False)].k)
              & set(comp[~comp.k.duplicated(keep=False)].k))

    # cause -> label, learned on strictly-matched TRAIN rows only; keep only pure causes
    jt = trL[trL.k.isin(strict)].merge(apL[["k", "causeofdeath", "GeneralCoD"]], on="k")
    ct = jt.groupby("causeofdeath").is_climate_sensitive.agg(["mean", "size"])
    cause_map = ct[(ct["mean"] == 0) | (ct["mean"] == 1)]["mean"].astype(int).to_dict()
    apL["label"] = apL.apply(lambda r: cause_map.get(r.causeofdeath,
                                                     GENERAL_RULE.get(r.GeneralCoD, np.nan)), axis=1)

    grp = apL.dropna(subset=["label"]).groupby("k").label.agg(["nunique", "mean"])
    lab_strict = {k: int(v) for k, v in grp[grp.index.isin(strict)]["mean"].items()}
    lab_routeA = {k: int(v) for k, v in
                  grp[(grp["nunique"] == 1) & (~grp.index.isin(strict))]["mean"].items()}
    label_map = {**lab_strict, **lab_routeA}

    log.info(f"  APHRC records {len(apL)} | strict keys {len(lab_strict)} | route-A keys {len(lab_routeA)}"
             f" | pure causes learned {len(cause_map)}")
    if not lab_strict:
        raise RuntimeError("zero strict matches -- check the APHRC file / date parsing")

15:30:40 | INFO    | ▶ linkage


15:30:40 | INFO    |   APHRC records 505 | strict keys 399 | route-A keys 103 | pure causes learned 64


15:30:40 | INFO    | ✔ linkage  (0.0s)


**Evidence that the target is a recode of cause of death.** The table below shows the broad cause
category versus the competition label on strictly-matched train rows. Every category is almost
perfectly pure.

In [10]:
purity = (jt.groupby("GeneralCoD").is_climate_sensitive.agg(n="size", positive_rate="mean")
            .sort_values("positive_rate", ascending=False))
purity

,n,positive_rate
GeneralCoD,,
Anaemia,7,1.0
"Communicable diseases, Perinatal and nutritional causes",163,1.0
Neonatal death,6,1.0
direct maternal causes,9,1.0
External causes,21,0.0
non communicable diseases,83,0.0


**Accuracy of each route on train rows where the truth is known.** The model alone is about 78%
accurate on these rows.

In [11]:
route_rows = []
for name, m in (("strict", lab_strict), ("route A", lab_routeA)):
    v = trL[trL.k.isin(m)].copy(); v["pred"] = v.k.map(m)
    route_rows.append(dict(route=name, train_rows=len(v),
                           accuracy=(v.pred == v.is_climate_sensitive).mean(),
                           test_rows=int(teL.k.isin(m).sum())))
route_table = pd.DataFrame(route_rows).set_index("route")
route_table.round(4)

,train_rows,accuracy,test_rows
route,,,
strict,289,1.000,110
route A,43,0.907,12


**Effect of the override on grouped-CV (diagnostic).** We apply the same override to the
out-of-fold blend on matched train rows. *Caveat:* the cause map is learned on these same train
rows, so this is mildly optimistic for strict rows. It shows the size of the effect at the train
coverage of 10.6%. Test coverage is higher, at 11.8%.

In [12]:
k_tr = trL.k.map(label_map)
hit_tr = k_tr.notna().values
ov = blend_oof.copy()
ov[:, hit_tr] = np.where(k_tr[hit_tr].values == 1, 0.999, 0.001)
per_seed_ov = np.array([combo(y, ov[s]) for s in range(ov.shape[0])])
override_cv = pd.DataFrame({
    "combo": [per_seed.mean(), per_seed_ov.mean()],
    "se": [per_seed.std(ddof=1) / np.sqrt(20), per_seed_ov.std(ddof=1) / np.sqrt(20)],
    "train_coverage": [0.0, hit_tr.mean()]},
    index=["model only (v8)", "model + APHRC override (v10)"])
override_cv.round(4)

,combo,se,train_coverage
model only (v8),0.8245,0.0003,0.0000
model + APHRC override (v10),0.8484,0.0003,0.1055


In [13]:
with stage("inference stage 2"):
    teL["label"] = teL.k.map(label_map)
    hit = teL.label.notna()
    log.info(f"  test rows matched: {hit.sum()} of {len(teL)} ({hit.mean():.1%})  "
             f"[strict {teL.k.isin(lab_strict).sum()} + route A {teL.k.isin(lab_routeA).sum()}]")

    sub = pd.read_csv(V8_PATH)                      # stage-1 output, read back as in the original run
    m = sub.merge(teL[["ID", "label"]], on="ID", how="left")
    h = m.label.notna()
    flips = int((m.loc[h, "TargetF1"].astype(int) != m.loc[h, "label"].astype(int)).sum())
    log.info(f"  model disagreed with the recorded cause on {flips} of {h.sum()} matched rows "
             f"({flips / h.sum():.1%})")

    m.loc[h, "TargetF1"] = m.loc[h, "label"].astype(int)
    m.loc[h, "TargetRAUC"] = np.where(m.loc[h, "label"] == 1, 0.999, 0.001)
    final = m[["ID", "TargetF1", "TargetRAUC"]].copy()
    final["TargetF1"] = final["TargetF1"].astype(int)
    check_submission(final, teL.ID)

    FINAL_PATH = OUT_DIR / "submission_v10.csv"
    final.to_csv(FINAL_PATH, index=False)
    log.info(f"  wrote {FINAL_PATH.relative_to(ROOT)}: n={len(final)} pos-rate {final.TargetF1.mean():.4f} "
             f"| coverage {hit.mean():.1%} | md5 {md5(FINAL_PATH)}")
final.head()

15:30:40 | INFO    | ▶ inference stage 2


15:30:40 | INFO    |   test rows matched: 122 of 1030 (11.8%)  [strict 110 + route A 12]


15:30:40 | INFO    |   model disagreed with the recorded cause on 39 of 122 matched rows (32.0%)


15:30:40 | INFO    |   wrote outputs/submission_v10.csv: n=1030 pos-rate 0.8136 | coverage 11.8% | md5 e5d3086d37ab847b5f975054cbb94286


15:30:40 | INFO    | ✔ inference stage 2  (0.0s)


,ID,TargetF1,TargetRAUC
0,ID_E760D84B,0,0.001000
1,ID_6EDEA907,1,0.761306
2,ID_B9FFC8D8,1,0.650793
3,ID_74C6C94E,1,0.603027
4,ID_0E02825D,1,0.800391


## 11. Reproducibility check against the submitted file

`reference/submission_v10.csv` is the exact file submitted to Zindi (ID `aymD7bTd`). The run
passes if the regenerated file is **byte-identical**, meaning the MD5 matches.

On a different OS or CPU architecture, tree libraries can differ in the last floating-point bits.
For that case the cell also reports the maximum absolute difference in `TargetRAUC`, the
`TargetF1` agreement and the AUC-rank correlation. On the original machine (macOS arm64) the
match is exact.

In [14]:
REF_PATH = REF_DIR / "submission_v10.csv"
if REF_PATH.exists():
    ref = pd.read_csv(REF_PATH)
    same_md5 = md5(FINAL_PATH) == md5(REF_PATH)
    repro = pd.Series({
        "md5 regenerated": md5(FINAL_PATH),
        "md5 reference": md5(REF_PATH),
        "byte-identical": same_md5,
        "same ID order": final.ID.equals(ref.ID),
        "TargetF1 agreement": (final.TargetF1.values == ref.TargetF1.values).mean(),
        "max |ΔTargetRAUC|": float(np.abs(final.TargetRAUC.values - ref.TargetRAUC.values).max()),
        "Spearman(TargetRAUC)": final.TargetRAUC.corr(ref.TargetRAUC, method="spearman"),
    })
    (log.info if same_md5 else log.warning)(
        "  REPRODUCED submission_v10.csv BYTE-FOR-BYTE" if same_md5 else
        "  output is NOT byte-identical to the reference -- see table for the size of the difference")
else:
    repro = pd.Series({"reference": "not found -- skipped"})
    log.warning(f"  {REF_PATH} not found; skipping comparison")
repro.to_frame("value")

15:30:40 | INFO    |   REPRODUCED submission_v10.csv BYTE-FOR-BYTE


,value
md5 regenerated,e5d3086d37ab847b5f975054cbb94286
md5 reference,e5d3086d37ab847b5f975054cbb94286
byte-identical,True
same ID order,True
TargetF1 agreement,1.0
max |ΔTargetRAUC|,0.0
Spearman(TargetRAUC),1.0


## 12. Run time

These are the wall-clock times for each stage of this notebook. The table below is filled in
when the notebook runs and is also saved to `outputs/runtime.json`.

Reference machine: Apple M5 (10 cores), 16 GB RAM, macOS, Python 3.12.13, `N_JOBS=10`.
Training is about 900 small tree models (3 members × 20 fold seeds × 5 folds × 3 model seeds).
The work parallelises across fold seeds, so run time scales roughly as `1/N_JOBS`. A single
core needs about 2 minutes. No GPU is used.

In [15]:
RUNTIME["TOTAL (notebook)"] = time.perf_counter() - T_START
rt = pd.Series(RUNTIME, name="seconds").round(2)
(OUT_DIR / "runtime.json").write_text(json.dumps(rt.to_dict(), indent=2))
rt.to_frame()

,seconds
extract,0.03
transform,0.01
load,0.03
train lgbm_reg,4.82
train cat_core,4.73
train xgb_v3,0.72
validate blend,0.04
inference stage 1,0.00
linkage,0.03
inference stage 2,0.01


## 13. Performance metrics

### Leaderboard
| Submission | Public LB | Private LB | Notes |
|---|---|---|---|
| **`submission_v10.csv`** (this notebook) | **0.849879848** | **0.850718556** (rank #5) | final model + APHRC override (122 rows) |
| `submission_v9.csv` | 0.849879848 | — | override with strict matches only (118 rows). Same public score; v10 adds route A |
| `submission_v8.csv` (stage 1 here) | 0.828728799 | — | model only |

### Cross-validation (repeated grouped CV, 20 untouched fold seeds)
These are computed live in §8 and §10. The values from the reference run were:

| | combo | F1 | AUC |
|---|---|---|---|
| `lgbm_reg` | 0.8249 ± 0.0004 | 0.8222 | 0.8303 |
| `cat_core` | 0.8217 ± 0.0004 | 0.8199 | 0.8271 |
| `xgb_v3` | 0.8220 ± 0.0003 | 0.8167 | 0.8284 |
| **Blend (1/3 each)** | **0.8245 ± 0.0003** | 0.8214 | 0.8312 |

### Linkage quality (on train rows with known truth)
| Route | Train rows | Accuracy | Test rows |
|---|---|---|---|
| strict | 289 | 100.0% | 110 |
| route A | 43 | 90.7% | 12 |
| (model alone on the same rows) | | ~78% | |

### Other metrics we tracked
* **CV → LB transfer.** v7 → v8 gave +0.0015 in CV and +0.0019 on the LB. v8 → v9 (the
  override) gave a simulated +0.023 and a realised +0.021 on the LB. The two closest-related
  submissions were used to check that the CV tracks the LB.
* **Positive rate of predictions.** This is 0.834 for stage 1 and 0.814 for the final file,
  against 0.651 in train. A deliberate high-recall operating point maximises F1.
* **Public-subset noise.** The SD of combo on a ~30% public subset is about 0.017. LB moves
  smaller than that were not used for decisions without CV support.

*ETL efficiency.* The whole extract/transform/load path runs in under 1 s. Input validation
covers file presence, schema, ID uniqueness, ID-set equality with the sample submission, NaN and
finiteness checks, and checksums.

## 14. Error handling and logging

**Logging.** A single `logging` logger (`v10`) writes timestamped records to the notebook output
and to **`outputs/run.log`**. Every stage is wrapped in `stage(...)`. That context manager logs
the start and end, records the duration for §12 and logs the full traceback if the stage fails.

**Fail-fast checks, by stage:**

| Stage | Check | Behaviour |
|---|---|---|
| Extract | Zindi files present | `FileNotFoundError` with instructions on where to put them |
| Extract | APHRC file present, or downloadable | auto-download, otherwise `RuntimeError` with the manual URL |
| Extract | APHRC checksum | **hard fail**, because a different file means different labels |
| Extract | Zindi checksums | **warning** (the host may re-export the data) |
| Extract | required columns, unique IDs, binary target, no NaNs, Test IDs = SampleSubmission IDs | `AssertionError` with a message |
| Transform | row counts preserved, all features finite | `AssertionError` |
| Linkage | APHRC dates parse 100% (six formats tried) | `AssertionError`. This prevents silent zero matches |
| Linkage | at least one strict match | `RuntimeError` |
| Inference | submission schema, ID order, binary `TargetF1`, `TargetRAUC` ∈ [0,1], no NaNs, `TargetF1 == (TargetRAUC ≥ 0.5)` | `AssertionError`. Runs on both the stage-1 and the final file |
| Output | MD5 against the reference submission | warning, with a numeric diff table (§11) |

## 15. Deployment, versioning and retraining

**Infrastructure.** This is batch inference on a single CPU machine. The only requirements are
Python 3.12 and the pinned packages in `requirements.txt`. There is no GPU, service or database.
The notebook *is* the deployable unit. It can run headless with
`jupyter nbconvert --execute`, from a scheduler (cron or Airflow) or in a container.

**New data in.** Add new records to `data/raw/Test.csv` with the same schema as the Zindi
`Test.csv` (`ID, zone, gender, deathdate, age, avg/max/min_temperature, precipitation, latitude,
longitude, location`). Then rerun the notebook. Note that the **REGIME features are
transductive**: they are computed over the train and test inputs together. Scoring a new batch
therefore means recomputing them with that batch included. This happens automatically when the
notebook reruns.

**Output interpretation.**
* `TargetRAUC` is a ranking score in [0, 1]. For model-scored rows it is the blended
  probability that the death is climate-sensitive. For linked rows it is 0.999 or 0.001, a
  near-certain value taken from the recorded cause of death.
* `TargetF1` is the 0/1 decision at threshold 0.5.
* The two columns always agree: `TargetF1 == (TargetRAUC ≥ 0.5)`.

**Versioning.** Inputs and outputs are pinned by MD5 (`MD5` dict in §3) and package versions by
`requirements.txt`. Each run writes `outputs/run.log`, `cv_metrics.csv` and `runtime.json`, so a
run can be traced back to its exact data, code and environment. We suggest tagging the repository
per released model (for example `v10`).

**Retraining.** Retrain when new labelled records arrive, at least yearly, because the labelling
regime drifts strongly over time (see §17). Retraining is the same notebook with an extended
`Train.csv`. If more APHRC verbal-autopsy records become available, replace
`data/external/IMHDSS_Climate_Mortality.csv`, update or remove its checksum in `MD5`, and rerun.
The full 5,054-record release is at the APHRC microdata portal (catalogue 165). The linkage code
is schema-driven and date-format-tolerant, so nothing else needs to change. With the full release,
coverage would rise from about 12% to an estimated 95% of rows.

## 16. Maintenance and monitoring

**What to monitor on each run:**
1. **Input drift.** Track the year distribution, the under-5 share and the record volume per
   year (the REGIME features). A new surveillance or coding regime, such as the 2018 shift,
   changes the base rate sharply. Alert if `y_infshare` or the annual volume moves more than
   about 3 SD from history.
2. **Linkage health.** Track the match coverage (currently 11.8%), the strict and route-A
   counts and the train accuracy per route (currently 100% and 90.7%). A sudden drop to 0
   matches means the APHRC export format changed. The date-format assertion is designed to catch
   that.
3. **Prediction health.** Track the positive rate (currently about 0.81) and the probability
   range. A shift of more than about 5 percentage points without a matching input shift points
   to a pipeline fault.
4. **Model quality.** Once labels for a new period arrive, compute combo, F1 and AUC on them and
   compare with CV (0.8245). Retrain if the drop is larger than about 0.01.

**Scaling.** Training cost is linear in rows × (fold seeds × folds × model seeds). For much larger
data, reduce `FOLD_SEEDS` for the final fit or refit once on the full data. During development,
a bagged-vs-full-data refit agreed on 99.4% of binary predictions, with correlation 0.9994. The
`joblib` parallelism scales across cores or, through a Dask or Ray backend, across machines. The
fingerprint join is a hash join and scales linearly. Features would move to partitioned Parquet.

**Lifecycle.** Keep the notebook, `requirements.txt`, the checksums and the reference submission
under version control. Treat any change to features, members or seeds as a new model version. It
must beat the previous version on the same untouched fold seeds by more than about 2 SE before
it ships.

## 17. Notes and insights for the host

1. **The target is almost a deterministic recode of the verbal-autopsy cause of death.** On
   the 289 strictly-linked train rows, the cause of death maps to the label with 100% accuracy
   (§10). The
   residual error of any model trained on the given columns is mostly irreducible from those
   columns. Access to the cause of death resolves it. For an operational system, the most
   valuable investment is linking to the VA records, not further climate-feature engineering.
2. **Strong labelling drift.** The positive rate falls from about 78% (2008) to about 50%
   (2021), and the infant share of records changes abruptly around 2018. This looks like a
   change in collection or coding practice rather than epidemiology. Time-based CV collapses
   (AUC about 0.45). Models must be told the era (`year`, the REGIME block).
3. **No village-level signal.** Across the 34 train villages with at least 30 records, the spread
   of village positive rates is *smaller* than binomial chance (ratio 0.87). Location features,
   spatial clustering, distance-to-city or lake, population density and village target-encoding
   all failed for this reason.
4. **Climate covariates did not add predictive value at this sample size.** This holds for the
   provided CHIRPS/ERA5/MODIS/SRTM features and for the extra NASA POWER and Open-Meteo humidity
   features we tried. The demographic core (age, gender, zone, year) carries the signal. This
   does not mean climate is irrelevant to these deaths. It means that, *given age and era*,
   same-day and 30/90-day weather at about 3k records cannot separate the classes further.
5. **Evaluation pitfalls we hit and fixed.** Early stopping on validation folds inflated CV by
   different amounts per configuration. Single-seed CV had an SE larger than most
   configuration differences. Both are fixed in §7, using fixed round counts and 20 repeated
   seeds with a reported SE.
6. **Tried and rejected:** using linked rows as extra training data (−0.0002 to −0.0156,
   worse with larger weights); fuzzy age matching ±1-2 years (77.8% accuracy, no better than the
   model); threshold re-optimisation (the optimum is exactly 0.5); pseudo-labelling; era-routed
   models; OOF-optimised blend weights (they overfit). Swapping CatBoost and XGBoost for
   ExtraTrees improved CV, but it was measured at train's base rate and lost 0.0046 on the LB.

## 18. External data and attribution

| Data | Source | Use |
|---|---|---|
| IMHDSS Climate & Mortality (verbal-autopsy 10% sample, 505 records) | APHRC, [huggingface.co/datasets/APHRC/Climate_and_Health](https://huggingface.co/datasets/APHRC/Climate_and_Health). Same sample mirrored on Harvard Dataverse `doi:10.7910/DVN/DHDNIC` | Record linkage and override (§10) |

External data is permitted in this challenge, as confirmed by the host. Under Zindi policy,
external data must be publicly available, under 1 GB and disclosed on the discussion forum. This
file is public (about 114 KB). *Acknowledgement (from the dataset card):* this work was
carried out with the aid of a grant from Lacuna Fund, an initiative co-founded by The Rockefeller
Foundation, Google.org, Canada's International Development Research Centre, and GIZ on behalf of
the German Federal Ministry for Economic Cooperation and Development (BMZ); and Wellcome.

No other external data, pretrained model or manual labelling is used by this submission.